# Notebook 07 — Knowledge Distillation

Knowledge distillation trains a small student model to mimic a large teacher model, producing a compact model that retains most of the teacher's performance.

In [ ]:
# !pip install transformers datasets torch

## 1. Distillation concepts

In [ ]:
# Teacher: large, accurate model (e.g., BERT-large)
# Student: small, fast model (e.g., DistilBERT)
#
# Training signal:
#   Hard loss: cross-entropy with true labels (standard classification loss)
#   Soft loss: KL divergence between teacher and student softmax distributions
#              using a temperature T > 1 to soften the distributions
#
# Total loss = alpha * hard_loss + (1 - alpha) * soft_loss * T^2
#
# Why T^2? The gradients of the soft loss scale as 1/T^2, so we multiply
# to keep gradients at the same scale regardless of temperature.

import torch
import torch.nn.functional as F

def distillation_loss(student_logits, teacher_logits, labels, T=4.0, alpha=0.5):
    soft_student = F.log_softmax(student_logits / T, dim=-1)
    soft_teacher = F.softmax(teacher_logits / T, dim=-1)
    soft_loss = F.kl_div(soft_student, soft_teacher, reduction="batchmean") * (T ** 2)
    hard_loss = F.cross_entropy(student_logits, labels)
    return alpha * hard_loss + (1 - alpha) * soft_loss

# Demo
student_logits = torch.randn(4, 2)
teacher_logits = torch.randn(4, 2)
labels = torch.tensor([0, 1, 0, 1])
loss = distillation_loss(student_logits, teacher_logits, labels)
print("Distillation loss:", loss.item())

## 2. Load teacher and student models

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

TEACHER_MODEL = "bert-base-uncased"
STUDENT_MODEL = "distilbert-base-uncased"

teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL)
teacher_model = AutoModelForSequenceClassification.from_pretrained(TEACHER_MODEL, num_labels=2)

student_tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL)
student_model = AutoModelForSequenceClassification.from_pretrained(STUDENT_MODEL, num_labels=2)

teacher_model.eval()  # teacher is frozen during distillation
for param in teacher_model.parameters():
    param.requires_grad = False

print("Teacher params:", sum(p.numel() for p in teacher_model.parameters()) // 1_000_000, "M")
print("Student params:", sum(p.numel() for p in student_model.parameters()) // 1_000_000, "M")

## 3. Distillation training loop

In [ ]:
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding
from torch.optim import AdamW

dataset = load_dataset("imdb", split="train[:1000]")

def preprocess(batch):
    return student_tokenizer(batch["text"], truncation=True, max_length=128)

tokenized = dataset.map(preprocess, batched=True, remove_columns=["text"])
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format("torch")

collator = DataCollatorWithPadding(tokenizer=student_tokenizer)
loader = DataLoader(tokenized, batch_size=16, collate_fn=collator)
optimizer = AdamW(student_model.parameters(), lr=2e-5)

for epoch in range(1):
    total_loss = 0
    for batch in loader:
        labels = batch.pop("labels")
        with torch.no_grad():
            teacher_out = teacher_model(**batch).logits
        student_out = student_model(**batch).logits
        loss = distillation_loss(student_out, teacher_out, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch 1 — avg loss: {total_loss / len(loader):.4f}")
print("Distillation training complete.")